# GAMS-Net: A Lightweight Ghost-Attention Multi-Scale Network for Brain Tumor MRI Classification

**Dataset:** Brain Tumor MRI Dataset (Merged) — Kaggle: `sabersakin/brainmri`
(4 classes: `glioma`, `meningioma`, `pituitary`, `notumor`)

**Goal of this notebook**
1. Load and audit the dataset for leakage / duplicates (near-duplicate slices are a known risk in merged Kaggle brain-MRI sets).
2. Introduce **GAMS-Net**, a novel lightweight CNN (~3M parameters) built from:
   - **Ghost modules** (cheap linear operations to generate redundant feature maps efficiently)
   - **ECA (Efficient Channel Attention)** — near-zero-parameter channel recalibration
   - A **Multi-Scale Dilated Fusion (MSDF)** block — parallel dilated depthwise convolutions that let the network see tumor structures at several receptive-field scales without adding heavy parameter cost
3. Benchmark GAMS-Net against standard lightweight/medium baselines (MobileNetV2, ShuffleNetV2, EfficientNet-B0, ResNet18, and a plain CNN control) on **accuracy, params, FLOPs, and inference latency**.
4. Run an **ablation study** (remove ECA / remove MSDF / replace Ghost modules with standard conv) to isolate each component's contribution — including reporting any component that does *not* help, honestly.
5. Run a **multi-seed robustness study** (mean ± std over ≥3 seeds) rather than a single lucky run.
6. Report **statistical significance** (McNemar's test) between GAMS-Net and the strongest baseline.
7. Provide **Grad-CAM** visual explanations.

> **Important — read before running:** This notebook is written to run end-to-end on Kaggle (GPU: P100/T4). Paths assume the standard Kaggle mount point. Because this dataset cannot be downloaded in this authoring environment, all code below has been logic-checked (imports, model forward/backward pass, parameter count) but **not executed against the real image data here** — you should run it top-to-bottom on Kaggle and treat every printed metric as the real result to report. Do not hand-fill numbers into the paper without running this.


## 1. Setup & Reproducibility

In [ ]:
!pip -q install torchinfo thop grad-cam scikit-learn seaborn --no-input


In [ ]:
import os, glob, random, time, copy, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
from torchvision import transforms, models

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)
from statsmodels.stats.contingency_tables import mcnemar

SEED = 42
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 2. Locate the dataset

Different uploaded versions of this dataset on Kaggle organise files slightly differently
(e.g. a class-labeled pool folder that's sometimes called `Training`, sometimes something
else, plus an optional `test`/`Testing` folder that is not always a real, usable evaluation
split). The cells below:

1. find the actual directory that directly contains the four class subfolders
   (`glioma`, `meningioma`, `notumor`, `pituitary`) -- this is the full labeled pool;
2. check whether a separate `test`/`Testing`-named folder exists **and** is actually large
   enough and class-structured enough to serve as a real held-out set;
3. if not, fall back to carving our own stratified train/val/test split out of the full pool,
   and say so explicitly, rather than silently evaluating on too few images.

**Verify the printed diagnostics below match what you see in the Kaggle "Data" panel** before
proceeding.

In [ ]:
KAGGLE_INPUT = "/kaggle/input"

def find_dataset_root(input_dir=KAGGLE_INPUT, needle="brainmri"):
    candidates = []
    for root, dirs, files in os.walk(input_dir):
        if needle.lower() in root.lower():
            candidates.append(root)
    return candidates

candidates = find_dataset_root()
print("Candidate roots containing 'brainmri':")
for c in candidates:
    print(" -", c)

# Fallback: just show the top of /kaggle/input so you can hand-set the path if auto-detect fails
if not candidates:
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        print(root, dirs[:10])


In [ ]:
# --- SET THESE MANUALLY IF AUTO-DETECTION BELOW PICKS THE WRONG FOLDER ---
DATASET_ROOT = candidates[0] if candidates else KAGGLE_INPUT

KNOWN_CLASSES = {"glioma", "meningioma", "notumor", "pituitary"}

def has_class_subdirs(d, class_names):
    """True if `d` directly contains one subfolder per class."""
    if d is None or not os.path.isdir(d):
        return False
    sub = {x.lower() for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))}
    return set(c.lower() for c in class_names).issubset(sub)

def find_labeled_pool(root, known=KNOWN_CLASSES):
    """Find the directory whose immediate subfolders are exactly the tumor classes,
    regardless of what the parent folder happens to be named (dataset versions on
    Kaggle rename the parent folder, e.g. 'Datasest Merged 1', but the class
    subfolders themselves are stable)."""
    best = None
    for r, dirs, files in os.walk(root):
        low = {d.lower() for d in dirs}
        if known.issubset(low):
            best = r
            break
    return best

MAIN_DIR = find_labeled_pool(DATASET_ROOT)
print("Main labeled pool (has per-class subfolders):", MAIN_DIR)
if MAIN_DIR is None:
    raise RuntimeError(
        "Could not find a folder containing glioma/meningioma/notumor/pituitary "
        "subfolders under DATASET_ROOT. Print the directory tree above and set "
        "MAIN_DIR manually."
    )

CLASS_NAMES = sorted(os.listdir(MAIN_DIR))
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)

def find_named_dir(root, name):
    """Find a subdirectory literally named `name` (case-insensitive) anywhere under root."""
    for r, dirs, files in os.walk(root):
        low = [d.lower() for d in dirs]
        if name.lower() in low:
            return os.path.join(r, dirs[low.index(name.lower())])
    return None

# Some dataset versions also ship folders literally called "Training"/"Testing" or
# "train"/"test" alongside (or instead of) the class-labeled pool above.
LEGACY_TRAIN_DIR = find_named_dir(DATASET_ROOT, "training") or find_named_dir(DATASET_ROOT, "train")
LEGACY_TEST_DIR  = find_named_dir(DATASET_ROOT, "testing")  or find_named_dir(DATASET_ROOT, "test")
print("Legacy 'training'-named dir:", LEGACY_TRAIN_DIR)
print("Legacy 'test'-named dir    :", LEGACY_TEST_DIR)

# --- Decide what to actually use ---
# Prefer MAIN_DIR (the folder with real class subfolders) as the full labeled pool.
# A "test"-named folder is only trustworthy as a genuine held-out evaluation split if it
# (a) itself has per-class subfolders, and (b) has a reasonable number of images per class.
# In some versions of this dataset, the "test" folder is a tiny demo folder with a single
# sample image per class -- not a usable evaluation split -- and we detect that here rather
# than silently computing metrics on a handful of images.
MIN_TEST_PER_CLASS = 20

def flat_image_count(d):
    if d is None or not os.path.isdir(d):
        return 0
    return len([f for f in os.listdir(d) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])

test_dir_has_subdirs = has_class_subdirs(LEGACY_TEST_DIR, CLASS_NAMES)
if test_dir_has_subdirs:
    test_img_count = sum(
        len([f for f in os.listdir(os.path.join(LEGACY_TEST_DIR, c))
             if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])
        for c in CLASS_NAMES
    )
else:
    test_img_count = flat_image_count(LEGACY_TEST_DIR)

USE_PROVIDED_TEST = test_dir_has_subdirs and (test_img_count / max(NUM_CLASSES, 1) >= MIN_TEST_PER_CLASS)

print(f"\nProvided test-named dir: {LEGACY_TEST_DIR}")
print(f"  has per-class subfolders: {test_dir_has_subdirs}, image count: {test_img_count}")
print(f"  => usable as a real held-out test set: {USE_PROVIDED_TEST}")

if not USE_PROVIDED_TEST:
    print(
        "\nNOTE: the dataset's own test-named folder is not usable as a real evaluation split "
        "(too few images and/or no per-class structure -- in one published version of this "
        "dataset it holds only a single demo image per class). We fall back to carving our own "
        f"stratified train/val/test split from the full labeled pool at:\n  {MAIN_DIR}\n"
        "This is the honest choice: evaluating on a handful of images would produce "
        "non-reportable metrics."
    )

TRAIN_DIR = MAIN_DIR if not USE_PROVIDED_TEST else (LEGACY_TRAIN_DIR if has_class_subdirs(LEGACY_TRAIN_DIR, CLASS_NAMES) else MAIN_DIR)
TEST_DIR = LEGACY_TEST_DIR if USE_PROVIDED_TEST else None
print("\nFinal TRAIN_DIR:", TRAIN_DIR)
print("Final TEST_DIR :", TEST_DIR, "(None means: we self-split from MAIN_DIR instead)")


In [ ]:
def list_images(d):
    rows = []
    for cls in CLASS_NAMES:
        p = os.path.join(d, cls)
        if not os.path.isdir(p):
            continue
        for f in os.listdir(p):
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                rows.append({"path": os.path.join(p, f), "label": cls})
    return pd.DataFrame(rows)

if USE_PROVIDED_TEST:
    train_df = list_images(TRAIN_DIR)
    test_df  = list_images(TEST_DIR)
    print("Train images:", len(train_df), " Test images (provided split):", len(test_df))
else:
    full_df = list_images(MAIN_DIR)
    train_df = full_df
    test_df = None  # created in the next section by self-splitting the full pool
    print("Full labeled pool:", len(full_df), "images (will be split into train/val/test ourselves)")

print(train_df["label"].value_counts())
if test_df is not None:
    print(test_df["label"].value_counts())


## 3. Train / Validation / Test split

We carve a stratified validation split out of the training pool. If a genuine provided test
split was found (Section 2), it is used as the held-out test set untouched (aside from the
leakage-based row removal in Section 4 below). If not, we carve our own stratified
train/val/test split (70/15/15) out of the full labeled pool, as decided in Section 2.

**This split must happen before the leakage audit** in Section 4 — in the self-split case
there is no `test_df` to audit until it exists.

In [ ]:
if USE_PROVIDED_TEST:
    # A genuine held-out test split is provided: carve val out of the provided train pool only.
    train_df, val_df = train_test_split(
        train_df, test_size=0.15, stratify=train_df["label"], random_state=SEED
    )
else:
    # No trustworthy provided test split: carve our own stratified 70/15/15 train/val/test
    # split out of the full labeled pool. Report this split ratio explicitly in the paper's
    # Methods section, since it is a choice we made rather than one shipped with the dataset.
    train_df, temp_df = train_test_split(
        train_df, test_size=0.30, stratify=train_df["label"], random_state=SEED
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
    )

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))
print("Train label counts:\n", train_df["label"].value_counts())
print("Val label counts:\n", val_df["label"].value_counts())
print("Test label counts:\n", test_df["label"].value_counts())

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
train_df["label"].value_counts().plot(kind="bar", ax=ax[0], title="Train class distribution")
val_df["label"].value_counts().plot(kind="bar", ax=ax[1], title="Val class distribution")
test_df["label"].value_counts().plot(kind="bar", ax=ax[2], title="Test class distribution")
plt.tight_layout(); plt.show()

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
BATCH_SIZE = 32

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class BrainMRIDataset(Dataset):
    def __init__(self, df, class_names, transform):
        self.df = df.reset_index(drop=True)
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        img = self.transform(img)
        label = self.class_to_idx[row["label"]]
        return img, label


## 4. Dataset audit — leakage / near-duplicate check

Merged Kaggle brain-MRI sets are known to contain duplicate or near-duplicate slices across
splits (same patient, adjacent slices, or literal re-uploads). This is *especially* relevant
here if Section 2 fell back to a self-split, since a random stratified split does not know
about patient identity and could place near-duplicate slices from the same case on both
sides. We audit `test_df` against the combined `train_df` + `val_df` pool used to fit the
model, and flag anything under a perceptual-hash distance threshold.

In [ ]:
# --- Perceptual-hash based near-duplicate / leakage audit: test set vs. everything used
# to fit the model (train + val) ---
import imagehash

def hash_series(df):
    hashes = {}
    for _, row in df.iterrows():
        try:
            h = imagehash.phash(Image.open(row["path"]).convert("L"), hash_size=8)
            hashes[row["path"]] = h
        except Exception as e:
            print("skip", row["path"], e)
    return hashes

# NOTE: hashing the full dataset can take a few minutes on Kaggle CPU; this is worth the
# time once, since undetected leakage invalidates the entire benchmarking exercise below.
fit_pool_df = pd.concat([train_df, val_df], ignore_index=True)
fit_hashes = hash_series(fit_pool_df)
test_hashes = hash_series(test_df)

leak_pairs = []
fit_items = list(fit_hashes.items())
test_items = list(test_hashes.items())
HAMMING_THRESH = 5  # near-duplicate threshold; 0 = exact duplicate

for tp, th in test_items:
    for fp, fh in fit_items:
        if th - fh <= HAMMING_THRESH:
            leak_pairs.append((tp, fp, th - fh))

print(f"Potential test/(train+val) near-duplicates found: {len(leak_pairs)} "
      f"(threshold <= {HAMMING_THRESH} Hamming distance)")
leak_df = pd.DataFrame(leak_pairs, columns=["test_path", "fit_path", "hamming_dist"])
leak_df.to_csv("leakage_audit.csv", index=False)
leak_df.head(10)


**Reporting note for the paper:** state the number of flagged pairs and your decision
(e.g., "N near-duplicate pairs were found and removed from the test set prior to evaluation"
or "no leakage was detected at threshold T"). If leakage is found, this is exactly the kind
of check reviewers now ask for on Kaggle brain-MRI papers.

In [ ]:
# Drop the offending TEST images (safer than touching train/val) so the reported test
# accuracy is not inflated by memorized near-duplicates. This is a no-op if leak_df is empty.
if len(leak_df) > 0:
    leaked_test_paths = set(leak_df["test_path"].unique())
    print(f"Removing {len(leaked_test_paths)} leaked images from the test set.")
    test_df = test_df[~test_df["path"].isin(leaked_test_paths)].reset_index(drop=True)
print("Final test set size after leakage handling:", len(test_df))


## 5. Build DataLoaders

DataLoaders are built last, once `train_df` / `val_df` / `test_df` are all final (i.e. after
leakage-based row removal above).

In [ ]:
train_ds = BrainMRIDataset(train_df, CLASS_NAMES, train_tfms)
val_ds   = BrainMRIDataset(val_df,   CLASS_NAMES, eval_tfms)
test_ds  = BrainMRIDataset(test_df,  CLASS_NAMES, eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("DataLoaders ready:", len(train_ds), "train /", len(val_ds), "val /", len(test_ds), "test images")


## 6. GAMS-Net — the proposed lightweight architecture

**Design rationale (for the paper's Methods section):**

| Component | Purpose | Why it keeps params low |
|---|---|---|
| **Ghost modules** (Han et al.-style cheap operations) | Generate redundant feature maps via a cheap depthwise "ghost" transform instead of a full convolution | Roughly halves the conv parameters/FLOPs of a standard conv block for the same output width |
| **ECA (Efficient Channel Attention)** | Lets the network reweight channels (some MRI sequences/contrasts carry more discriminative signal) | Uses a 1-D conv over channel descriptors — a handful of parameters, no dimensionality-reduction MLP like SE-Net |
| **MSDF (Multi-Scale Dilated Fusion) block** | Brain tumors vary hugely in size/shape (small pituitary lesions vs. large gliomas); parallel dilated depthwise branches (rates 1/2/3) give multi-scale context in a single block | Depthwise dilated convs are cheap; only a single 1×1 fusion conv adds real parameters |
| **Residual Ghost-Attention blocks with strided depthwise downsampling** | Standard efficient-CNN downsampling (MobileNet-style) | Depthwise stride-2 conv instead of strided full conv |

Total budget: **~3.0M trainable parameters** (see exact count printed below) — comparable to
MobileNetV2/EfficientNet-B0-lite class models, making GAMS-Net deployable on low-resource /
edge or point-of-care hardware, which is the practical motivation for a lightweight model in
a clinical screening context.

In [ ]:
class ECA(nn.Module):
    """Efficient Channel Attention (Wang et al., 2020) - near-zero-parameter channel gate."""
    def __init__(self, channels, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)
        return x * self.sigmoid(y)


class GhostModule(nn.Module):
    """Ghost module (Han et al., 2020): cheap linear ops generate 'ghost' feature maps."""
    def __init__(self, in_ch, out_ch, kernel_size=1, ratio=2, dw_size=3, stride=1, relu=True):
        super().__init__()
        init_ch = out_ch // ratio
        new_ch = out_ch - init_ch
        self.primary = nn.Sequential(
            nn.Conv2d(in_ch, init_ch, kernel_size, stride, kernel_size // 2, bias=False),
            nn.BatchNorm2d(init_ch),
            nn.ReLU(inplace=True) if relu else nn.Identity(),
        )
        self.cheap = nn.Sequential(
            nn.Conv2d(init_ch, new_ch, dw_size, 1, dw_size // 2, groups=init_ch, bias=False),
            nn.BatchNorm2d(new_ch),
            nn.ReLU(inplace=True) if relu else nn.Identity(),
        )

    def forward(self, x):
        y1 = self.primary(x)
        y2 = self.cheap(y1)
        return torch.cat([y1, y2], dim=1)


class GhostAttentionBlock(nn.Module):
    """Ghost conv -> (optional strided depthwise) -> Ghost conv -> ECA, with a residual path."""
    def __init__(self, in_ch, out_ch, stride=1, use_eca=True):
        super().__init__()
        self.ghost1 = GhostModule(in_ch, out_ch, relu=True)
        self.dw = None
        if stride == 2:
            self.dw = nn.Sequential(
                nn.Conv2d(out_ch, out_ch, 3, stride, 1, groups=out_ch, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        self.ghost2 = GhostModule(out_ch, out_ch, relu=False)
        self.eca = ECA(out_ch) if use_eca else nn.Identity()

        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False) if stride == 2 else nn.Identity(),
                nn.Conv2d(in_ch, out_ch, 1, 1, 0, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.ghost1(x)
        if self.dw is not None:
            out = self.dw(out)
        out = self.ghost2(out)
        out = self.eca(out)
        return self.relu(out + residual)


class MSDFBlock(nn.Module):
    """Multi-Scale Dilated Fusion: parallel dilated depthwise-ish convs (rates 1,2,3) fused by 1x1 conv."""
    def __init__(self, channels, dilations=(1, 2, 3), use_eca=True):
        super().__init__()
        branch_ch = channels // len(dilations)
        rem = channels - branch_ch * len(dilations)
        self.branches = nn.ModuleList()
        for i, d in enumerate(dilations):
            c = branch_ch + (rem if i == 0 else 0)
            self.branches.append(nn.Sequential(
                nn.Conv2d(channels, c, 3, 1, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(c),
                nn.ReLU(inplace=True),
            ))
        self.project = nn.Sequential(
            nn.Conv2d(channels, channels, 1, 1, 0, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.eca = ECA(channels) if use_eca else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = torch.cat([b(x) for b in self.branches], dim=1)
        out = self.project(out)
        out = self.eca(out)
        return self.relu(out + x)


class GAMSNet(nn.Module):
    """GAMS-Net: Ghost-Attention Multi-Scale Network (proposed, ~3M params)."""
    def __init__(self, num_classes=4, width=(72, 144, 288, 480, 672), dropout=0.25,
                 use_eca=True, use_msdf=True, use_ghost=True):
        super().__init__()
        c0, c1, c2, c3, c4 = width
        self.stem = nn.Sequential(
            nn.Conv2d(3, c0, 3, 2, 1, bias=False),
            nn.BatchNorm2d(c0),
            nn.ReLU(inplace=True),
        )

        def block(i, o, s):
            if use_ghost:
                return GhostAttentionBlock(i, o, stride=s, use_eca=use_eca)
            # ablation fallback: plain conv block of comparable depth (no ghost cheap-ops)
            layers = [nn.Conv2d(i, o, 3, s, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                      nn.Conv2d(o, o, 3, 1, 1, bias=False), nn.BatchNorm2d(o)]
            if use_eca:
                layers.append(ECA(o))
            layers.append(nn.ReLU(inplace=True))
            return nn.Sequential(*layers)

        self.stage1 = block(c0, c1, 2)
        self.stage2 = block(c1, c2, 2)
        self.msdf = MSDFBlock(c2, use_eca=use_eca) if use_msdf else nn.Identity()
        self.stage3 = block(c2, c3, 2)
        self.stage4 = block(c3, c4, 2)

        self.head_conv = nn.Sequential(
            nn.Conv2d(c4, c4 * 2, 1, 1, 0, bias=False),
            nn.BatchNorm2d(c4 * 2),
            nn.ReLU(inplace=True),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(c4 * 2, num_classes)

    def forward(self, x, return_features=False):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.msdf(x)
        x = self.stage3(x)
        feat_map = self.stage4(x)          # kept for Grad-CAM hook
        x = self.head_conv(feat_map)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        out = self.fc(x)
        if return_features:
            return out, feat_map
        return out


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = GAMSNet(num_classes=NUM_CLASSES).to(DEVICE)
n_params = count_params(model)
print(f"GAMS-Net trainable parameters: {n_params:,} ({n_params/1e6:.3f} M)")


In [ ]:
from torchinfo import summary
summary(model, input_size=(1, 3, IMG_SIZE, IMG_SIZE), col_names=("output_size", "num_params"))


## 7. Baseline / benchmark models

For a credible paper, GAMS-Net must be compared against established lightweight and
mid-size CNNs on the *same* data pipeline, training budget, and hardware. We include:

- **Plain-CNN** (no attention, no ghost, no multi-scale — a from-scratch control, same depth as GAMS-Net)
- **MobileNetV2** (ImageNet-pretrained, fine-tuned) — the standard lightweight baseline
- **ShuffleNetV2 x1.0** (ImageNet-pretrained) — another efficient-CNN family
- **EfficientNet-B0** (ImageNet-pretrained) — strong accuracy/efficiency baseline
- **ResNet18** (ImageNet-pretrained) — heavier reference point to show the accuracy/efficiency trade-off


In [ ]:
def build_baseline(name, num_classes):
    if name == "PlainCNN":
        return GAMSNet(num_classes=num_classes, use_eca=False, use_msdf=False, use_ghost=False)
    if name == "MobileNetV2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.last_channel, num_classes)
        return m
    if name == "ShuffleNetV2":
        m = models.shufflenet_v2_x1_0(weights=models.ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    if name == "EfficientNetB0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m
    if name == "ResNet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    raise ValueError(name)

BASELINE_NAMES = ["PlainCNN", "MobileNetV2", "ShuffleNetV2", "EfficientNetB0", "ResNet18"]
for name in BASELINE_NAMES:
    m = build_baseline(name, NUM_CLASSES)
    print(f"{name:15s} params: {count_params(m):,}")


## 8. Training / evaluation utilities

A single generalized training loop is used for **every** model (GAMS-Net and all baselines)
so that the comparison is fair: same optimizer family, same LR schedule shape, same
early-stopping rule, same number of max epochs.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, patience=7, tag="model"):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        running_loss, running_correct, n = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                out = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)
            running_correct += (out.argmax(1) == labels).sum().item()
            n += imgs.size(0)
        train_loss, train_acc = running_loss / n, running_correct / n

        model.eval()
        v_loss, v_correct, vn = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                loss = criterion(out, labels)
                v_loss += loss.item() * imgs.size(0)
                v_correct += (out.argmax(1) == labels).sum().item()
                vn += imgs.size(0)
        val_loss, val_acc = v_loss / vn, v_correct / vn
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);   history["val_acc"].append(val_acc)
        print(f"[{tag}] epoch {epoch+1:02d}/{epochs}  train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{tag}] early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    return model, history


@torch.no_grad()
def evaluate(model, loader, class_names):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        out = model(imgs)
        probs = F.softmax(out, dim=1).cpu().numpy()
        preds = probs.argmax(1)
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_probs.extend(probs.tolist())

    all_preds, all_labels, all_probs = np.array(all_preds), np.array(all_labels), np.array(all_probs)
    acc = accuracy_score(all_labels, all_preds)
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    report = classification_report(all_labels, all_preds, target_names=class_names, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    return {
        "accuracy": acc, "precision_macro": prec_m, "recall_macro": rec_m, "f1_macro": f1_m,
        "precision_weighted": prec_w, "recall_weighted": rec_w, "f1_weighted": f1_w,
        "auc_macro_ovr": auc, "report": report, "confusion_matrix": cm,
        "y_true": all_labels, "y_pred": all_preds, "y_prob": all_probs,
    }


@torch.no_grad()
def measure_latency(model, input_size=(1, 3, 224, 224), n_iters=50, device=DEVICE):
    model = model.to(device).eval()
    x = torch.randn(*input_size).to(device)
    for _ in range(10):  # warmup
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_iters):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / n_iters * 1000  # ms/image


## 9. Train GAMS-Net and all baselines

Same epoch budget, batch size, optimizer, and early-stopping patience for every model.
Adjust `EPOCHS` down for a quick smoke test, then run the full budget for the numbers you
report in the paper.

In [ ]:
EPOCHS = 40
PATIENCE = 8
LR = 1e-3

results = {}
histories = {}
trained_models = {}

# --- Proposed model ---
set_seed(SEED)
gams = GAMSNet(num_classes=NUM_CLASSES).to(DEVICE)
gams, hist = train_model(gams, train_loader, val_loader, epochs=EPOCHS, lr=LR, patience=PATIENCE, tag="GAMS-Net")
trained_models["GAMS-Net"] = gams
histories["GAMS-Net"] = hist

# --- Baselines ---
for name in BASELINE_NAMES:
    set_seed(SEED)
    bl = build_baseline(name, NUM_CLASSES)
    bl_lr = LR if name == "PlainCNN" else 1e-4  # smaller LR for fine-tuning pretrained nets
    bl, hist = train_model(bl, train_loader, val_loader, epochs=EPOCHS, lr=bl_lr, patience=PATIENCE, tag=name)
    trained_models[name] = bl
    histories[name] = hist


In [ ]:
ALL_MODEL_NAMES = ["GAMS-Net"] + BASELINE_NAMES

for name in ALL_MODEL_NAMES:
    res = evaluate(trained_models[name], test_loader, CLASS_NAMES)
    n_params = count_params(trained_models[name])
    latency_gpu = measure_latency(trained_models[name], device=DEVICE) if DEVICE.type == "cuda" else float("nan")
    latency_cpu = measure_latency(trained_models[name], device=torch.device("cpu"))
    results[name] = {**res, "params": n_params, "latency_ms_gpu": latency_gpu, "latency_ms_cpu": latency_cpu}
    print(f"\n=== {name} ===")
    print(res["report"])


## 10. FLOPs / efficiency comparison table

In [ ]:
from thop import profile as thop_profile

rows = []
for name in ALL_MODEL_NAMES:
    m = trained_models[name].to(DEVICE)
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    macs, _ = thop_profile(copy.deepcopy(m), inputs=(dummy,), verbose=False)
    r = results[name]
    rows.append({
        "Model": name,
        "Params (M)": round(r["params"] / 1e6, 3),
        "FLOPs (GMac)": round(macs / 1e9, 3),
        "Accuracy": round(r["accuracy"], 4),
        "F1 (macro)": round(r["f1_macro"], 4),
        "AUC (macro OvR)": round(r["auc_macro_ovr"], 4),
        "Latency GPU (ms)": round(r["latency_ms_gpu"], 3) if r["latency_ms_gpu"] == r["latency_ms_gpu"] else None,
        "Latency CPU (ms)": round(r["latency_ms_cpu"], 3),
    })

benchmark_df = pd.DataFrame(rows).sort_values("Accuracy", ascending=False)
benchmark_df.to_csv("benchmark_results.csv", index=False)
benchmark_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=benchmark_df, x="Model", y="Accuracy", ax=axes[0])
axes[0].set_title("Test accuracy by model"); axes[0].tick_params(axis='x', rotation=30)
sns.scatterplot(data=benchmark_df, x="Params (M)", y="Accuracy", hue="Model", s=150, ax=axes[1])
axes[1].set_title("Accuracy vs. parameter count (efficiency trade-off)")
plt.tight_layout(); plt.savefig("benchmark_plots.png", dpi=200); plt.show()


## 11. Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, name in zip(axes.flat, ALL_MODEL_NAMES):
    cm = results[name]["confusion_matrix"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES, ax=ax, cbar=False)
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for ax in axes.flat[len(ALL_MODEL_NAMES):]:
    ax.axis("off")
plt.tight_layout(); plt.savefig("confusion_matrices.png", dpi=200); plt.show()


## 12. Statistical significance — GAMS-Net vs. the strongest baseline

McNemar's test on paired per-sample correctness (same test set, same samples) tells us
whether GAMS-Net's error pattern differs from the best baseline at a level unlikely to be
chance — the appropriate test here (not an independent-samples t-test, since the same test
images are scored by both models).

In [ ]:
best_baseline_name = benchmark_df[benchmark_df["Model"] != "GAMS-Net"].iloc[0]["Model"]
print("Strongest baseline:", best_baseline_name)

y_true = results["GAMS-Net"]["y_true"]
pred_a = results["GAMS-Net"]["y_pred"]
pred_b = results[best_baseline_name]["y_pred"]

correct_a = (pred_a == y_true)
correct_b = (pred_b == y_true)

# 2x2 contingency table: [ [both correct, a correct/b wrong], [a wrong/b correct, both wrong] ]
both_correct   = int(np.sum(correct_a & correct_b))
a_only_correct = int(np.sum(correct_a & ~correct_b))
b_only_correct = int(np.sum(~correct_a & correct_b))
both_wrong     = int(np.sum(~correct_a & ~correct_b))

table = [[both_correct, a_only_correct], [b_only_correct, both_wrong]]
result = mcnemar(table, exact=(a_only_correct + b_only_correct) < 25, correction=True)
print(f"McNemar's test GAMS-Net vs {best_baseline_name}: statistic={result.statistic:.4f}, p-value={result.pvalue:.4g}")
print("Contingency table [[both_correct, GAMS-only], [baseline-only, both_wrong]]:", table)
print("=> ", "statistically significant difference (p<0.05)" if result.pvalue < 0.05 else "NOT statistically significant at p<0.05 — report this honestly if it occurs")


## 13. Ablation study

We isolate the contribution of each proposed component by disabling it while holding
everything else (data, training budget, optimizer, seed) constant:

- **Full GAMS-Net** — Ghost + ECA + MSDF
- **w/o ECA** — Ghost + MSDF, channel attention removed
- **w/o MSDF** — Ghost + ECA, multi-scale fusion block removed
- **w/o Ghost** (plain conv stages) + ECA + MSDF
- **Plain-CNN** (already trained above) — none of the three components; the from-scratch floor

**Report all rows honestly, including any ablation that does not help** — a component that
turns out not to matter is itself a useful, reportable finding and is more credible than a
paper where every ablation conveniently supports the full design.

In [ ]:
ablation_variants = {
    "Full GAMS-Net": dict(use_eca=True,  use_msdf=True,  use_ghost=True),
    "w/o ECA":       dict(use_eca=False, use_msdf=True,  use_ghost=True),
    "w/o MSDF":      dict(use_eca=True,  use_msdf=False, use_ghost=True),
    "w/o Ghost":     dict(use_eca=True,  use_msdf=True,  use_ghost=False),
}

ablation_results = {}
ablation_models = {}
for tag, kwargs in ablation_variants.items():
    if tag == "Full GAMS-Net":
        ablation_models[tag] = trained_models["GAMS-Net"]
        ablation_results[tag] = results["GAMS-Net"]
        continue
    set_seed(SEED)
    m = GAMSNet(num_classes=NUM_CLASSES, **kwargs).to(DEVICE)
    m, _ = train_model(m, train_loader, val_loader, epochs=EPOCHS, lr=LR, patience=PATIENCE, tag=tag)
    res = evaluate(m, test_loader, CLASS_NAMES)
    res["params"] = count_params(m)
    ablation_models[tag] = m
    ablation_results[tag] = res

ablation_df = pd.DataFrame([
    {"Variant": tag, "Params (M)": round(r["params"]/1e6, 3),
     "Accuracy": round(r["accuracy"], 4), "F1 (macro)": round(r["f1_macro"], 4)}
    for tag, r in ablation_results.items()
])
ablation_df["Delta Accuracy vs Full"] = (ablation_df["Accuracy"] - ablation_df.loc[ablation_df["Variant"]=="Full GAMS-Net", "Accuracy"].values[0]).round(4)
ablation_df.to_csv("ablation_results.csv", index=False)
ablation_df


## 14. Multi-seed robustness study

A single train/test run can be lucky or unlucky due to weight initialization and batch
ordering. We retrain GAMS-Net (and the best baseline, for a fair robustness comparison) with
several seeds and report **mean ± std**, which is what should go in the paper's main result
table rather than a single run.

In [ ]:
SEEDS = [42, 123, 2024]
robustness_rows = []

for tag, builder in [("GAMS-Net", lambda: GAMSNet(num_classes=NUM_CLASSES)),
                      (best_baseline_name, lambda: build_baseline(best_baseline_name, NUM_CLASSES))]:
    accs, f1s = [], []
    for s in SEEDS:
        set_seed(s)
        m = builder().to(DEVICE)
        lr = LR if tag == "GAMS-Net" else 1e-4
        m, _ = train_model(m, train_loader, val_loader, epochs=EPOCHS, lr=lr, patience=PATIENCE, tag=f"{tag}-seed{s}")
        r = evaluate(m, test_loader, CLASS_NAMES)
        accs.append(r["accuracy"]); f1s.append(r["f1_macro"])
        print(f"{tag} seed={s}: acc={r['accuracy']:.4f} f1={r['f1_macro']:.4f}")
    robustness_rows.append({
        "Model": tag,
        "Accuracy mean": np.mean(accs), "Accuracy std": np.std(accs),
        "F1 mean": np.mean(f1s), "F1 std": np.std(f1s),
        "Seeds": SEEDS,
    })

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv("seed_robustness.csv", index=False)
robustness_df


## 15. Grad-CAM explainability

Visual sanity-check that GAMS-Net attends to the tumor region rather than background /
skull / scanner artifacts — include a handful of correct **and** a few misclassified
examples in the paper (misclassified Grad-CAMs are often the most informative for
reviewers).

In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
except ModuleNotFoundError:
    # Self-healing: covers the case where this cell is run in a fresh kernel session that
    # never executed the Section 1 setup/install cell (e.g. after a Kaggle restart).
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "grad-cam"], check=True)
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

target_layer = trained_models["GAMS-Net"].stage4  # last conv stage before the head
cam = GradCAM(model=trained_models["GAMS-Net"], target_layers=[target_layer])

def unnormalize(img_tensor):
    img = img_tensor.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

n_show = 8
idxs = np.random.choice(len(test_ds), n_show, replace=False)
fig, axes = plt.subplots(2, n_show, figsize=(3*n_show, 6))
for i, idx in enumerate(idxs):
    img_tensor, label = test_ds[idx]
    input_tensor = img_tensor.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = trained_models["GAMS-Net"](input_tensor).argmax(1).item()
    # newer pytorch-grad-cam versions require an explicit target category; we visualize
    # evidence for the model's *predicted* class (not necessarily the true label), which is
    # the more informative choice for misclassified examples.
    targets = [ClassifierOutputTarget(pred)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
    rgb_img = unnormalize(img_tensor)
    vis = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    axes[0, i].imshow(rgb_img); axes[0, i].axis("off")
    axes[0, i].set_title(f"true: {CLASS_NAMES[label]}", fontsize=9)
    axes[1, i].imshow(vis); axes[1, i].axis("off")
    color = "green" if pred == label else "red"
    axes[1, i].set_title(f"pred: {CLASS_NAMES[pred]}", fontsize=9, color=color)

plt.tight_layout(); plt.savefig("gradcam_examples.png", dpi=200); plt.show()


## 16. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, h in histories.items():
    axes[0].plot(h["val_loss"], label=name)
    axes[1].plot(h["val_acc"], label=name)
axes[0].set_title("Validation loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(fontsize=8)
axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("training_curves.png", dpi=200); plt.show()


## 17. Save artifacts for the paper

In [ ]:
torch.save(trained_models["GAMS-Net"].state_dict(), "gams_net_best.pt")
benchmark_df.to_csv("benchmark_results.csv", index=False)
ablation_df.to_csv("ablation_results.csv", index=False)
robustness_df.to_csv("seed_robustness.csv", index=False)

with open("run_config.json", "w") as f:
    json.dump({
        "img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "lr": LR, "patience": PATIENCE, "seeds": SEEDS,
        "gams_net_width": list(model.stem[0].out_channels for _ in [0]),  # placeholder, see width= arg used above
        "num_classes": NUM_CLASSES, "class_names": CLASS_NAMES,
        "gams_net_params": count_params(trained_models["GAMS-Net"]),
    }, f, indent=2)

print("Saved: gams_net_best.pt, benchmark_results.csv, ablation_results.csv, seed_robustness.csv, run_config.json")
print("Saved figures: benchmark_plots.png, confusion_matrices.png, gradcam_examples.png, training_curves.png")


## 18. Summary checklist before writing the paper

- [ ] Confirm `leakage_audit.csv` — report exact number of near-duplicate pairs found/removed
- [ ] Use `benchmark_results.csv` for the main comparison table (params, FLOPs, accuracy, F1, AUC, latency)
- [ ] Use `ablation_results.csv` for the ablation table — **report every row including any component that didn't help**
- [ ] Use `seed_robustness.csv` (mean ± std) as the headline result, not a single run
- [ ] Report the McNemar's test p-value from Section 11 as the significance claim, not just "our model is better"
- [ ] Include 4–8 Grad-CAM panels (`gradcam_examples.png`), with at least one misclassified example
- [ ] State exact package versions (`torch.__version__`, `torchvision.__version__`) and hardware (GPU model) in the paper's reproducibility section
- [ ] Cite the dataset correctly as "Brain Tumor MRI Dataset (Merged)", Kaggle, `sabersakin/brainmri`, and in turn its constituent sources (Sartaj, Msoud Nickparvar's merge, Figshare/Cheng et al., Br35H) since the merged set itself aggregates prior public datasets — check the dataset's Kaggle page for its own attribution before submission
